In [ ]:
!pip install pymorphy2 svgling spacy-udpipe -q

### Немного о том, как можно работать с синтаксисом

In [ ]:
import nltk
from nltk.grammar import *

Мы немноготкасались чанкинга на первом семинаре, давайте теперь о нем поподробнее. В nltk есть несколько модулей, которые принимают на вход некоторые правила, основанные на составляющих, и дальше выводят грамматику предложения на основе правил. В принципе, разница этих модулей только в самом процессе высчитывания грамматики, результаты одинаковые, так что можно сильно не задумываться.

#### 1. Можно получать разбор предложения
В случае, если разбор неоднозначный, библиотека сама высчитывает возможные варианты.

In [ ]:
el_grammar = nltk.CFG.fromstring(
"""
S -> NP VP
PP -> P NP
NP -> Det N | NP PP | 'I'
VP -> V NP | VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
"""
)

sent = ['I', 'shot', 'an', 'elephant', 'in', 'my', 'pajamas']
parser = nltk.EarleyChartParser(el_grammar)
for tree in parser.parse(sent):
    print(tree)

In [ ]:
trees = list(parser.parse(sent))
trees[0]

In [ ]:
trees[1]

#### 2. Можно посмотреть логику вычисления грамматики

Звездочка ставится тогда, когда не найдено то, что должно стоять после (какой фрагмент грамматики). Завершается тогда, когда после звездочки больше нет символов. Находит символ, потом ищется ближайший подходящий под правило и тд

In [ ]:
parser = nltk.EarleyChartParser(el_grammar, trace=1) # trace - глубина анализа работы парсера
for tree in parser.parse(sent):
    print(tree)

In [ ]:
parser = nltk.EarleyChartParser(el_grammar, trace=2)
for tree in parser.parse(sent):
    print(tree)

Проблемы:
- нельзя делать правила вида "NP -> 'New York'", можно только в одно слово, например, "New_York"

__Задание 1:__
Как будет выглядеть правило для следующего предложения?

In [ ]:
sent = 'очень красивое яблоко лежало на столе'

el_grammar = nltk.CFG.fromstring(
"""
SOME RULE
"""
)

parser = nltk.EarleyChartParser(el_grammar)
for tree in parser.parse(sent.split()):
    print(tree)

In [ ]:
trees = list(parser.parse(sent.split()))
trees[0]

### Зависимости

Аналогичные правила, только немного другой модуль: DependencyGrammar.

In [ ]:
dep_rules = """
    'shot' -> 'I' | 'elephant' | 'in' | 'morning'
    'elephant' -> 'an' | 'in'
    'in' -> 'pajamas'
    'pajamas' -> 'my'
"""

In [ ]:
dep_grammar = nltk.DependencyGrammar.fromstring(dep_rules)
pdp = nltk.ProjectiveDependencyParser(dep_grammar)
for tree in pdp.parse(sent):
    print(tree)

__Задание 2:__ Сделайте похожее правило для русского языка

In [ ]:
dep_rules = """
SOME RULE
"""

dep_grammar = nltk.DependencyGrammar.fromstring(dep_rules)
pdp = nltk.ProjectiveDependencyParser(dep_grammar)

sent = 'опять наши дети ушли утром в школу'.split()
for tree in pdp.parse(sent):
    print(tree)

## Spacy и что он умеет автоматически

Сегодня разберем новую модель, которую делали на основе корпусов connl-u. Считается, что она чуть хуже модели, например, stanza, но зато быстрая.

In [ ]:
import spacy_udpipe
from spacy import displacy
spacy_udpipe.download("ru")

nlp_udpipe = spacy_udpipe.load("ru")

#### Рисовать зависимости

In [ ]:
text = "Мама пришла со мной в платье"

doc = nlp_udpipe(text)
displacy.render(doc, style='dep')

#### Выводить информацию о роли слова в предложении
Хотя скорее не слова, а токена

In [ ]:
text = "Женщина в платье увидела мужчину, который нёс желтый зонтик с пятнышками"

doc = nlp_udpipe(text)
for token in doc:
    print(token.text, token.lemma_, token.pos_, token.dep_)

In [ ]:
# показать как может работать, если перед "который" будет не мужчина
displacy.render(doc, style='dep')

__Задание 3__: как можно эту штуку сломать?

In [ ]:
text = "Предложение, которое может сломать"

doc = nlp_udpipe(text)
displacy.render(doc, style='dep')

### Можно работать с анализом предложения

In [ ]:
from spacy.symbols import nsubj, VERB, obj, NOUN, amod, ADJ

In [ ]:
token = doc[0]
print(token)
print(token.dep, token.dep_)
print(token.pos, token.pos_)
print(type(token.dep), type(token.dep_))

In [ ]:
print(token.dep==nsubj)
print(token.dep_=='nsubj')

#### Достать пару прилагательное-существительное

In [ ]:
nps = set()
for possible_adj in doc:
    if possible_adj.dep == amod and possible_adj.head.pos == NOUN:
        nps.add((possible_adj, possible_adj.head))
print(nps)

nps = set()
for possible_adj in doc:
    if possible_adj.pos == ADJ and possible_adj.head.pos == NOUN:
        nps.add((possible_adj, possible_adj.head))
print(nps)

#### Достать пару подлежащее-сказуемое

__Задание:__ как можно аналогично достать пару подлежащее-сказуемое? Какие роли/части речи нам нужно проверить? Почему?

In [ ]:
nps = set()
# YOUR CODE HERE #

print(nps)

#### Достать пару главное-зависимое

In [ ]:
nps = set()
for possible_subject in doc:
    children = [child for child in possible_subject.children]
    for child in children:
        print(possible_subject, child.dep_, child.text)

#### Если очень хочется, то можно создавать свои датасеты, на основе вывода парсера.

In [ ]:
import pandas as pd

dep_df = pd.DataFrame(columns=['text', 'dep', 'pos', 'lefts', 'rights', 'ancestor'])

root = [token for token in doc if token.head == token][0]
subject = list(root.lefts)[0]
for i, descendant in enumerate(doc):
    dep_df.loc[i] = [descendant.text, descendant.dep_, descendant.pos_, ','.join([left.text for left in descendant.lefts]), ','.join([right.text for right in descendant.rights]), ','.join([ancestor.text for ancestor in descendant.ancestors])]

In [ ]:
dep_df

#### Доставать поддеревья

In [ ]:
nps = set()
for possible_subject in doc:
    if possible_subject.dep in [nsubj, obj] and possible_subject.pos==NOUN:
        print(possible_subject, list(possible_subject.subtree))

#### Можно доставать начало и конец под-дерева и объединять его в один токен.

In [ ]:
text = "Женщина в платье увидела мужчину, который нёс желтый зонтик с пятнышками"

doc = nlp_udpipe(text)
span = doc[doc[9].left_edge.i : doc[9].right_edge.i+1]

with doc.retokenize() as retokenizer:
    retokenizer.merge(span)
for token in doc:
    print(token.text, token.pos_, token.dep_, token.head.text)

## Практика: учимся использовать spacy

Можно попробовать делать аугментации. Что это?

Представим, что у нас есть хорошая модель для того, чтобы доставать имена из текста. Это полезно, например, для задачи анонимизации текста. Но есть проблема: какие-то имена всё-таки не распознаются. Попробуем добавить эти имена в данные.

In [ ]:
!python -m spacy download ru_core_news_sm -q

In [ ]:
import spacy
import random

Загрузим модель и придумаем немного данных с именами.

In [ ]:
nlp = spacy.load('ru_core_news_sm')
data = [
    'Оля постоянно опаздывала на уроки',
    'Почему Миша постоянно хотел есть?',
    'А ты знал, что Коля выучил новый фокус?'
]

Логика такая:
- spacy умеет находить имена самостоятельноо и помечать их лейблом PER
- давайте будем их находить и вырезать из текста, заменяя своим именем
Вопросы:
- где лежат в spacy сущности
- можно ли просто заменить сущность с помощью str.replace?
- как можно случайно достать имя из списка?
- как можно сказать spacy, что добавленное слово имеет лейбл PER?

In [ ]:
female_list = [
    'блум',
    'стелла',
    'текна'
]
male_list = [
    'крош',
    'ёжик'
]

new_texts = []
name_starts = []
for text in data:
    doc = nlp(text)
    # Ищем сущность, находим её начало и конец
    # Заменяем в тексте

all_docs = []
for text, (start, name) in zip(new_texts, name_starts):
    doc = nlp(text)
    # Пишем лейбл для нового имени в тексте
    name_ent = # Что нам нужно #
    doc.set_ents([name_ent], default="unmodified")
    all_docs.append(doc)

In [ ]:
for i in range(len(new_texts)):
    print('Before:')
    doc_old = nlp(new_texts[i])
    print(doc_old)
    for ent in doc_old.ents:
        print(ent, ent.label_)

    print('After:')
    doc_new = all_docs[i]
    print(doc_new)
    for ent in doc_new.ents:
        print(ent, ent.label_)

Хочется ещё наверное, чтобы согласование было? А spacy это умеет?

In [ ]:
import re
from pymorphy2 import MorphAnalyzer
morph = MorphAnalyzer()

- Как можно достать из токена информацию о его морфологических характеристиках с помощью spacy?
- Если мы знаем, что есть ошибка в анализе, как можно его заменить?

Так как по сути, мы сразу знаем род имени, то можем на основе наших знаний поправить spacy, если он ошибается.

In [ ]:
final_texts = []
for doc in all_docs:
    new_text = doc.text
    start, name = name_starts[0]
    genders = {'Fem': 'femn', 'Masc': 'masc'}

    for token in doc:
        if token.text == name:
            token_morph = token.morph
            # поправить род для токена
            # YOUR CODE #

            # поменять согласование глагола
            if token.dep == nsubj and token.head.pos == VERB:
                gen = token.morph.get('Gender')[0]
                verb = token.head
                # YOUR CODE HERE #

                new_text = doc.text[:verb_start] + verb_parsed + doc.text[verb_end:]

    final_texts.append(new_text)